In [0]:
%sql

CREATE VOLUME databrickskrishna.bronze.autovol

**Autoloader**

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", "/Volumes/databrickskrishna/bronze/autovol/destination/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .load("/Volumes/databrickskrishna/bronze/autovol/raw/")
    # addNewColumns - adds the cols directly to the df, do only if you're confident if you need them all
    # rescue  - if new file has 20 more or 100 more cols, it creates a single column and create a dictionary
    # Once rescued you'll understand which columns to use

###### **Rescue**

In [0]:
df.writeStream.format("delta")\
.outputMode("append")\
.option("checkpointLocation", "/Volumes/databrickskrishna/bronze/autovol/destination/checkpoint/")\
.trigger(once = True)\
.start("/Volumes/databrickskrishna/bronze/autovol/destination/data")

In [0]:
df = spark.read.format("delta")\
.load("/Volumes/databrickskrishna/bronze/autovol/destination/data/")

display(df)

###### **addNewColumns**

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", "/Volumes/databrickskrishna/bronze/autovol/destination/checkpoint/")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/databrickskrishna/bronze/autovol/raw/")
    # addNewColumns - adds the cols directly to the df, do only if you're confident if you need them all
    # rescue  - if new file has 20 more or 100 more cols, it creates a single column and create a dictionary
    # Once rescued you'll understand which columns to use

In [0]:
df.writeStream.format("delta")\
.outputMode("append")\
.option("checkpointLocation", "/Volumes/databrickskrishna/bronze/autovol/destination/checkpoint/")\
.trigger(once = True)\
.option("mergeSchema", True)\
.start("/Volumes/databrickskrishna/bronze/autovol/destination/data")